In [1]:
!pip install pandas scikit-learn spacy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json

files = [
    r"D:\Climate_Intelligent_System\data\annotated\mayukha_annotations.json",
    r"D:\Climate_Intelligent_System\data\annotated\harshini_annotations.json",
    r"D:\Climate_Intelligent_System\data\annotated\ashley_annotations.json"
]

merged = []
for file in files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)
        merged.extend(data)

with open(r"D:\Climate_Intelligent_System\data\processed\merged_annotations.json", "w", encoding="utf-8") as f:
    json.dump(merged, f, indent=2)

print(f"Total tasks merged: {len(merged)}")

Total tasks merged: 263


In [3]:
#BIO CONVERSION
import json

def convert_to_bio(merged_file, output_file):
    with open(merged_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    bio_sentences = []
    skipped = 0
    
    for task in data:
        text = task["data"]["text"]
        words = text.split()
        tags = ["O"] * len(words)
        
        annotations = task.get("annotations", [])
        if not annotations:
            skipped += 1
            continue
            
        results = annotations[0].get("result", [])
        
        for entity in results:
            value = entity.get("value", {})
            start = value.get("start", 0)
            end = value.get("end", 0)
            label = value.get("labels", ["O"])[0]
            
            char_count = 0
            entity_started = False
            for i, word in enumerate(words):
                word_start = char_count
                word_end = char_count + len(word)
                if word_start >= start and word_end <= end + 1:
                    if not entity_started:
                        tags[i] = f"B-{label}"
                        entity_started = True
                    else:
                        tags[i] = f"I-{label}"
                char_count += len(word) + 1
        
        sentence = "\n".join([f"{word}\t{tag}" for word, tag in zip(words, tags)])
        bio_sentences.append(sentence)
    
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n\n".join(bio_sentences))
    
    print(f"Converted: {len(bio_sentences)} sentences")
    print(f"Skipped (no annotations): {skipped}")
    print(f"Saved to: {output_file}")

convert_to_bio(
    r"D:\Climate_Intelligent_System\data\processed\merged_annotations.json",
    r"D:\Climate_Intelligent_System\data\processed\climate_bio.txt"
)

Converted: 263 sentences
Skipped (no annotations): 0
Saved to: D:\Climate_Intelligent_System\data\processed\climate_bio.txt


In [4]:
import random

with open(r"D:\Climate_Intelligent_System\data\processed\climate_bio.txt", "r", encoding="utf-8") as f:
    sentences = f.read().split("\n\n")

# Remove empty sentences
sentences = [s.strip() for s in sentences if s.strip()]

random.seed(42)  # For reproducibility
random.shuffle(sentences)

total = len(sentences)
train_end = int(total * 0.8)
dev_end = int(total * 0.9)

train = sentences[:train_end]
dev = sentences[train_end:dev_end]
test = sentences[dev_end:]

for split, name in [(train, "train"), (dev, "dev"), (test, "test")]:
    path = rf"D:\Climate_Intelligent_System\data\processed\climate_{name}.txt"
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n\n".join(split))
    print(f"{name}: {len(split)} sentences → saved")

print(f"\nTotal: {total} sentences split into train/dev/test")

train: 210 sentences → saved
dev: 26 sentences → saved
test: 27 sentences → saved

Total: 263 sentences split into train/dev/test


In [5]:
# Previewing first 30 lines of BIO file to verify it looks correct
with open(r"D:\Climate_Intelligent_System\data\processed\climate_bio.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

print("First 30 lines of BIO output:")
print("".join(lines[:30]))

First 30 lines of BIO output:
Sea	B-Climate_Driver
level	I-Climate_Driver
rise	I-Climate_Driver
(SLR)	O
and	O
increased	O
urbanisation	B-Human_Activity
of	O
coastal	B-Geo_Location
areas	I-Geo_Location
have	O
exacerbated	O
coastal	O
flood	B-Env_Event
threats,	O
making	O
them	O
even	O
more	O
severe	O
in	O
important	O
cultural	B-Ecosystem
sites.	I-Ecosystem
In	O
this	O
context,	O
the	O
role	O
of	O



In [7]:
import json
from sklearn.metrics import cohen_kappa_score

def extract_token_labels(file, task_ids):
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    labels = {}
    for task in data:
        task_id = task["id"]
        if task_id not in task_ids:
            continue
            
        text = task["data"]["text"]
        words = text.split()
        tags = ["O"] * len(words)
        
        annotations = task.get("annotations", [])
        if not annotations:
            continue
        results = annotations[0].get("result", [])
        
        for entity in results:
            value = entity.get("value", {})
            start = value.get("start", 0)
            end = value.get("end", 0)
            label = value.get("labels", ["O"])[0]
            
            char_count = 0
            entity_started = False
            for i, word in enumerate(words):
                word_start = char_count
                word_end = char_count + len(word)
                if word_start >= start and word_end <= end + 1:
                    if not entity_started:
                        tags[i] = f"B-{label}"
                        entity_started = True
                    else:
                        tags[i] = f"I-{label}"
                char_count += len(word) + 1
        
        labels[task_id] = tags
    return labels

# Overlapping task IDs between Mayukha and Harshini
shared_ids = list(range(151, 201))  # tasks 151-200

mayukha_labels = extract_token_labels(
    r"D:\Climate_Intelligent_System\data\annotated\mayukha_annotations.json",
    shared_ids
)
harshini_labels = extract_token_labels(
    r"D:\Climate_Intelligent_System\data\annotated\harshini_annotations.json",
    shared_ids
)

print(f"Mayukha tasks found: {len(mayukha_labels)}")
print(f"Harshini tasks found: {len(harshini_labels)}")

# Find common tasks
common_tasks = set(mayukha_labels.keys()) & set(harshini_labels.keys())
print(f"Common tasks: {len(common_tasks)}")

# Flatten into two lists
all_mayukha = []
all_harshini = []

for task_id in sorted(common_tasks):
    m = mayukha_labels[task_id]
    h = harshini_labels[task_id]
    min_len = min(len(m), len(h))
    all_mayukha.extend(m[:min_len])
    all_harshini.extend(h[:min_len])

print(f"Total tokens compared: {len(all_mayukha)}")

# Calculate kappa
kappa = cohen_kappa_score(all_mayukha, all_harshini)
print(f"\nCohen's Kappa: {kappa:.4f}")

if kappa >= 0.8:
    print("Excellent agreement! Proceed to SciBERT training.")
elif kappa >= 0.6:
    print("Good agreement. Minor schema clarifications recommended.")
elif kappa >= 0.4:
    print("Moderate agreement. Review disagreements before training.")
else:
    print("Low agreement.")

Mayukha tasks found: 50
Harshini tasks found: 50
Common tasks: 50
Total tokens compared: 13635

Cohen's Kappa: 0.2412
Low agreement.
